<img src="../../images/logo/compactlogo.svg" width="200" alt="QENS">

# Tutorial 1: Quickstart — Your First QEC Simulation

This notebook walks you through the core QENS workflow in five steps:
1. Create a QEC code
2. Choose a noise model
3. Choose a decoder
4. Run a Monte Carlo simulation
5. Visualize results

**Prerequisites:** `pip install qens`

In [ ]:
import numpy as np
import qens
from qens import (
    RepetitionCode, SurfaceCode,
    DepolarizingError, BitFlipError,
    LookupTableDecoder, MWPMDecoder,
    NoisySampler, ThresholdExperiment,
    draw_lattice,
)
from qens.core.types import PauliOp

print(f"QENS version: {qens.__version__}")

## Step 1: Create a QEC Code

QENS provides three built-in code families:
- `RepetitionCode(distance)` — 1D bit-flip code, great for learning
- `SurfaceCode(distance)` — 2D toric/planar code, the leading near-term candidate
- `ColorCode(distance)` — 2D triangular color code, supports transversal gates

Start with the repetition code — simple, fast, and easy to visualize.

In [ ]:
code = RepetitionCode(distance=5)

print(f"Code type:       {type(code).__name__}")
print(f"Distance:        {code.code_distance}")
print(f"Data qubits:     {code.num_data_qubits}")
print(f"Ancilla qubits:  {code.num_ancilla_qubits}")
print(f"Stabilizers:     {len(code.stabilizer_generators())}")

A distance-5 repetition code has 5 data qubits and 4 ancilla qubits (one per stabilizer). It can correct up to **2 bit-flip errors**.

## Step 2: Understand the Stabilizer Structure

Each stabilizer is a Pauli operator that commutes with all other stabilizers. Measuring it reveals information about errors without disturbing the encoded logical qubit.

In [ ]:
stabs = code.stabilizer_generators()
print(f"Number of stabilizers: {len(stabs)}")
print()
for i, s in enumerate(stabs):
    print(f"Stabilizer {i}: type={s.stabilizer_type}, qubits={s.qubits}, pauli={s.pauli_string}")

## Step 3: Manually Inject Errors and Measure Syndromes

Before running a full Monte Carlo simulation, let's manually inject an error and see what syndrome it produces.

In [ ]:
# Inject a single X error on qubit 2
error = np.zeros(code.num_data_qubits, dtype=np.uint8)
error[2] = PauliOp.X

syndrome = code.compute_syndrome(error)

print(f"Error:    {error}   (X on qubit 2)")
print(f"Syndrome: {syndrome}")
print()
print("Triggered stabilizers:", list(np.nonzero(syndrome)[0]))

In [ ]:
# Now inject a two-qubit error
error2 = np.zeros(code.num_data_qubits, dtype=np.uint8)
error2[1] = PauliOp.X
error2[3] = PauliOp.X

syndrome2 = code.compute_syndrome(error2)
print(f"Error:    {error2}   (X on qubits 1 and 3)")
print(f"Syndrome: {syndrome2}")
print("Triggered stabilizers:", list(np.nonzero(syndrome2)[0]))

## Step 4: Decode the Syndrome

A decoder takes a syndrome and returns the most likely correction. For small codes, the `LookupTableDecoder` gives the optimal (minimum-weight) correction.

In [ ]:
decoder = LookupTableDecoder(code)
decoder.precompute()  # Builds the lookup table (optional — auto-called on first decode)

result = decoder.decode(syndrome)
print(f"Syndrome:   {syndrome}")
print(f"Correction: {result.correction}")
print(f"Table hit:  {result.metadata['table_hit']}")

In [ ]:
# Verify: residual = error * correction should have no logical component
from qens.utils.pauli_algebra import pauli_string_multiply

residual, _ = pauli_string_multiply(error, result.correction)
is_logical = code.is_logical_error(residual)

print(f"Original error: {error}")
print(f"Correction:     {result.correction}")
print(f"Residual:       {residual}")
print(f"Logical error:  {is_logical}")

## Step 5: Run a Monte Carlo Simulation

In practice, we run thousands of shots to estimate the **logical error rate** at a given physical noise level.

In [ ]:
noise = BitFlipError(p=0.05)   # 5% bit-flip rate

sampler = NoisySampler(seed=42)
sim_result = sampler.run(code, noise, decoder, shots=5_000)

print(f"Shots:              {sim_result.num_shots}")
print(f"Logical errors:     {sim_result.logical_error_count}")
print(f"Logical error rate: {sim_result.logical_error_rate:.4f}")

## Step 6: Visualize the Code Lattice

The `draw_lattice` function renders a 2D diagram of the code geometry.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Remove this line in Jupyter to display inline

surface = SurfaceCode(distance=3)
fig = draw_lattice(surface, title="Surface Code d=3")
fig.show()

## Step 7: Sweep Error Rates (Threshold Plot)

The **threshold** is the physical error rate below which larger codes perform better. Below threshold, increasing the code distance exponentially suppresses the logical error rate.

In [ ]:
from qens.viz.stats import plot_threshold

experiment = ThresholdExperiment(
    code_class=RepetitionCode,
    distances=[3, 5, 7],
    physical_error_rates=[0.01, 0.03, 0.05, 0.08, 0.10, 0.12, 0.15],
    noise_model_factory=lambda p: BitFlipError(p=p),
    decoder_class=LookupTableDecoder,
    shots_per_point=2_000,
    seed=123,
)

threshold_result = experiment.run()

fig = plot_threshold(threshold_result, title="Repetition Code Threshold (Lookup Decoder)")
fig.show()

The crossing point of the curves is the **threshold error rate** — for the repetition code under bit-flip noise, this is 50% (each distance curve crosses the others near $p = 0.5$).

## Summary

| Step | API |
|------|-----|
| Create code | `RepetitionCode(distance=5)` |
| Inject error | `np.array(..., dtype=np.uint8)` with `PauliOp` values |
| Measure syndrome | `code.compute_syndrome(error)` |
| Decode | `decoder.decode(syndrome)` |
| Monte Carlo | `NoisySampler(seed=42).run(code, noise, decoder, shots=N)` |
| Threshold sweep | `ThresholdExperiment(...).run()` |

**Next:** [Tutorial 2 — Syndrome Extraction Deep Dive](02_syndrome_extraction.ipynb)